[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Files, Paths and Formats](https://johnfisher-ai.github.io/Python-Visual-Guides/files-paths-and-formats.html)

# Encodings


## What you will be able to do

Read a file whose encoding you were told wrongly, recognize the garbled text that results, and
explain why it happened rather than guessing at fixes.


## The idea

### The problem

Someone sends you a CSV. You read it, and where names should be you find `CafÃ©` and `naÃ¯ve`.
Or the read fails outright with a message about a byte in position 4,127. Nothing in your code
is wrong, and re-downloading the file changes nothing.

Or a subtler version: you read a spreadsheet export, ask for the column called `name`, and get a
`KeyError` on a file that plainly has a column called `name`.

All of these are the same problem, and it is worth being able to name it rather than trying
encodings until one works.

### Characters and bytes

> A file on disk contains **bytes**: numbers from 0 to 255. It does not contain text.
>
> An **encoding** is a rule for turning characters into bytes and back. Writing text applies the
> rule; reading applies it in reverse.
>
> **Decoding with a different rule than was used to encode does not fail.** It produces
> different characters, and often produces them silently.

That last sentence is the whole notebook. There is no marker inside a plain text file saying
which encoding was used. The program reading it has to be told, or has to guess.

### UTF-8, and why it is the answer

**UTF-8** can represent every character in every writing system, and it has one property that
made it win: the first 128 characters are encoded exactly as ASCII always did. A file of plain
English is byte-for-byte identical in ASCII and UTF-8.

That is why a file can look fine for years and then break the first time someone types an
accented name.

Everything you write should be UTF-8. The problem is everything you read, which was written by
someone else, possibly on Windows, possibly by Excel, possibly in 2011.

### Mojibake

`CafÃ©` is not corruption and not a rendering problem. It is exactly what you get when bytes
encoded as UTF-8 are decoded as if they were Latin-1.

The word is Japanese, from *moji* meaning character and *bake* meaning change. It has a name
because it is common enough to need one.

Recognizing the pattern is useful: `Ã©`, `Ã¨`, `â€™` and `Ã¼` appearing where accented letters
belong all mean the same thing, and the fix is the same each time.

### Where you will meet this

Any file you did not create. The **CSV** and **Excel Files** notebooks both depend on this one,
because spreadsheet exports are where encoding problems reach most people.

### What this notebook covers

- Encoding and decoding, and why the same string is a different number of bytes each way
- What mojibake actually is, and repairing it
- `UnicodeDecodeError`, and reading the position it names
- The `errors` argument, and which of its options quietly lose data
- The byte order mark, and why an Excel CSV has an invisible character in its first column name
- Why `latin-1` never fails, and why that is a problem rather than a feature
- Three errors, plus one that reads a file successfully and gives you the wrong text

### A first look

Nothing to run yet.

```python
text = "café"

print(len(text))                        # 4 characters
print(text.encode("utf-8"))             # b'caf\xc3\xa9'  -> 5 bytes
print(text.encode("latin-1"))           # b'caf\xe9'      -> 4 bytes

print(text.encode("utf-8").decode("latin-1"))    # 'cafÃ©'
```

One string, two rules, different bytes. The last line applies the wrong rule on the way back,
produces `cafÃ©`, and raises nothing at all.


## Setup

Three imports and a folder to work in.

**Run this cell before the rest of the notebook.**


In [1]:
from pathlib import Path
import locale
import shutil

scratch = Path("scratch")
scratch.mkdir(exist_ok=True)

print("working in:", scratch, "->", scratch.exists())


working in: scratch -> True


## Worked examples

### The same characters, different bytes


In [2]:
text = "caf\u00e9"

print("characters:", len(text), "|", text)

for encoding in ["utf-8", "latin-1", "utf-16"]:
    encoded = text.encode(encoding)
    print(f"{encoding:<8} {encoded!r:<26} {len(encoded)} bytes")


characters: 4 | café
utf-8    b'caf\xc3\xa9'             5 bytes
latin-1  b'caf\xe9'                 4 bytes
utf-16   b'\xff\xfec\x00a\x00f\x00\xe9\x00' 10 bytes


Four characters, and three different byte sequences of three different lengths. None is more
correct than the others; they are answers to different questions.

`utf-8` used two bytes for the accented character and one for each of the others. `latin-1` used
one for everything, which it can do only because it has no way to represent most of the world's
characters. `utf-16` used two bytes each plus a two-byte marker at the front.

### Plain English is identical in ASCII and UTF-8


In [3]:
plain = "hello"

print(plain.encode("utf-8"))
print(plain.encode("ascii"))
print("identical:", plain.encode("utf-8") == plain.encode("ascii"))


b'hello'
b'hello'
identical: True


This compatibility is why UTF-8 won, and it is why encoding problems arrive late. A file of
English text is valid ASCII, valid UTF-8, and valid Latin-1 at the same time, and every program
reads it correctly. The first accented character is where the three stop agreeing.

### Mojibake, produced on purpose


In [4]:
on_disk = "caf\u00e9".encode("utf-8")

print("bytes in the file:  ", on_disk)
print("decoded as utf-8:   ", on_disk.decode("utf-8"))
print("decoded as latin-1: ", on_disk.decode("latin-1"))


bytes in the file:   b'caf\xc3\xa9'
decoded as utf-8:    café
decoded as latin-1:  cafÃ©


`cafÃ©`. No error, no warning, and a program that carries on with that value will store it,
display it and export it exactly as it is.

The two-byte UTF-8 sequence for `é` is `0xc3 0xa9`. Latin-1 maps every byte to one character, so
it read those two bytes as two characters: `Ã` and `©`. Nothing was corrupted; the bytes were
read correctly and interpreted by the wrong rule.

Because nothing was lost, the damage can be undone.


In [5]:
broken = on_disk.decode("latin-1")

print("broken:  ", broken)
print("repaired:", broken.encode("latin-1").decode("utf-8"))


broken:   cafÃ©
repaired: café


Encode it back with the wrong rule to recover the original bytes, then decode with the right
one. That works when the wrong encoding was one that never fails, which is the next section.

Reach for it when you have inherited a file that is already mojibake and cannot be regenerated.
Fixing the reading is better whenever the original is still available.


### UnicodeDecodeError, and what it tells you


In [6]:
latin_file = scratch / "latin.txt"
latin_file.write_bytes("caf\u00e9\n".encode("latin-1"))

print(latin_file.read_text(encoding="utf-8"))


UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe9 in position 3: invalid continuation byte

`'utf-8' codec can't decode byte 0xe9 in position 3: invalid continuation byte`.

Three useful things in one line. The codec it tried, the byte it choked on, and where. In a real
file the position tells you which line to look at, and the byte value often identifies the
actual encoding: `0xe9` is `é` in Latin-1.

This is the good outcome. The file was written in Latin-1, UTF-8 refused it, and you found out.


In [7]:
print(repr(latin_file.read_text(encoding="latin-1")))


'café\n'


### The errors argument

`errors` decides what happens when a byte cannot be decoded. The default is `strict`, which
raises.


In [8]:
data = "caf\u00e9".encode("utf-8")

for mode in ["ignore", "replace", "backslashreplace"]:
    print(f"{mode:<18} {data.decode('ascii', errors=mode)!r}")


ignore             'caf'
replace            'caf��'
backslashreplace   'caf\\xc3\\xa9'


| Value | Behavior | Use it |
|---|---|---|
| `strict` | raises `UnicodeDecodeError` | always, unless you have a reason |
| `replace` | inserts `\ufffd`, the replacement character | when you must proceed and want the damage visible |
| `ignore` | drops the byte | almost never |
| `backslashreplace` | writes the byte as `\xNN` | when debugging, to see what was actually there |

`ignore` is the one to be careful with. It does not report a problem, and the text it returns
looks perfectly ordinary.


In [9]:
print("ignore: ", repr(latin_file.read_text(encoding="utf-8", errors="ignore")))
print("replace:", repr(latin_file.read_text(encoding="utf-8", errors="replace")))
print("correct:", repr(latin_file.read_text(encoding="latin-1")))


ignore:  'caf\n'
replace: 'caf�\n'
correct: 'café\n'


`'caf\n'`. The accented character is gone, the string is a perfectly valid name, and nothing
downstream can tell that a character was dropped. A file with a thousand accented names read
this way produces a thousand subtly wrong values and no error.

`replace` at least leaves a mark you can search for.


### latin-1 never fails, which is worse than failing

Latin-1 maps all 256 possible byte values to characters. There is no byte sequence it can
reject.


In [10]:
utf8_bytes = "h\u00e9llo w\u00f6rld".encode("utf-8")

print("as utf-8:  ", utf8_bytes.decode("utf-8"))
print("as latin-1:", utf8_bytes.decode("latin-1"))


as utf-8:   héllo wörld
as latin-1: hÃ©llo wÃ¶rld


That is why `latin-1` is a bad guess to try when you do not know the encoding. It will always
succeed, so success tells you nothing.

If you must guess, try `utf-8` first: it rejects most byte sequences that are not UTF-8, so it
succeeding is real evidence.

### The byte order mark

This is the one that produces the strangest bug report in this guide.


In [11]:
excel_csv = scratch / "export.csv"
excel_csv.write_text("name,value\nada,1\n", encoding="utf-8-sig")

print("first bytes:", excel_csv.read_bytes()[:6])


first bytes: b'\xef\xbb\xbfnam'


`\xef\xbb\xbf` at the front. That is a **byte order mark**, a marker some programs write to say
"this is UTF-8". Excel writes one when saving a CSV as UTF-8, and so do several other Windows
tools.

Read with plain `utf-8`, it becomes an invisible character at the start of the file.


In [12]:
as_utf8 = excel_csv.read_text(encoding="utf-8")
as_sig = excel_csv.read_text(encoding="utf-8-sig")

print("utf-8:     ", repr(as_utf8.split("\n")[0]))
print("utf-8-sig: ", repr(as_sig.split("\n")[0]))


utf-8:      '\ufeffname,value'
utf-8-sig:  'name,value'


`'\ufeffname,value'` against `'name,value'`. The first column is not called `name`; it is called
`\ufeffname`, and the character is invisible in every display you will look at.


In [13]:
print("first column == 'name'?")
print("  read as utf-8:    ", as_utf8.split(",")[0] == "name")
print("  read as utf-8-sig:", as_sig.split(",")[0] == "name")


first column == 'name'?
  read as utf-8:     False
  read as utf-8-sig: True


That is the `KeyError` on a column you can see. When a spreadsheet export behaves this way, read
it with `encoding="utf-8-sig"`, which strips the mark if present and does nothing if it is not.

It is a safe default for any file that came from Excel.


### What Python uses when you do not say

`open` and `read_text` have a default encoding, and the default is not the same everywhere.


In [14]:
print("locale encoding on this machine:", locale.getencoding())


locale encoding on this machine: UTF-8


On this machine it is UTF-8. On many Windows installations it is `cp1252`, and the same script
reading the same file gives different results on the two.

That is the argument for **always passing `encoding=`**. It costs a few characters and removes
an entire category of "works on my machine".


In [15]:
good = scratch / "explicit.txt"

good.write_text("caf\u00e9\n", encoding="utf-8")
print(repr(good.read_text(encoding="utf-8")))


'café\n'


## Your turn

Six tasks. Write your answer in the cell under each and run it.

Try each one before you look at an answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/files-paths-and-formats/03-encodings-solutions.ipynb).

**1.** Print the number of characters in `"naïve"` and the number of bytes it becomes in `utf-8`
and in `latin-1`.


In [16]:
# your code here


**2.** Encode `"señor"` as UTF-8 and decode the result as Latin-1. Print what you get.


In [17]:
# your code here


**3.** Repair the string from task 2 back to `"señor"`.


In [18]:
# your code here


**4.** Write `"Zürich\n"` to a file as Latin-1, then try to read it as UTF-8 and print the
exception type and message instead of letting it stop the cell.


In [19]:
# your code here


**5.** Read the same file three ways: `errors="ignore"`, `errors="replace"`, and with the
correct encoding. Print all three with `repr`.


In [20]:
# your code here


**6.** Write `"id,name\n1,ada\n"` with `encoding="utf-8-sig"`, then read it back both ways and
print whether the first column equals `"id"` in each case.


In [21]:
# your code here


## Common errors

Each cell below is run on purpose so you can see the real message.

### UnicodeDecodeError: the file is not what you said it was


In [22]:
mixed = scratch / "mixed.txt"
mixed.write_bytes("Z\u00fcrich\n".encode("latin-1"))

mixed.read_text(encoding="utf-8")


UnicodeDecodeError: 'utf-8' codec can't decode byte 0xfc in position 1: invalid start byte

Read the position. In a file of any size, `position 1` against `position 40127` is the
difference between "the whole file is the wrong encoding" and "one row has something unusual
in it".


### UnicodeEncodeError: the encoding cannot represent the character

The same problem going the other way.


In [23]:
"\u4f60\u597d".encode("latin-1")


UnicodeEncodeError: 'latin-1' codec can't encode characters in position 0-1: ordinal not in range(256)

`'latin-1' codec can't encode characters in position 0-1`. Latin-1 has 256 slots and no room for
Chinese characters.

UTF-8 has no such limit, which is the other half of why it is the answer.


In [24]:
print("\u4f60\u597d".encode("utf-8"))


b'\xe4\xbd\xa0\xe5\xa5\xbd'


### The quiet one: the file read successfully and the text is wrong

This raises nothing at all.


In [25]:
source = scratch / "names.txt"
source.write_bytes("Bj\u00f6rk\nRen\u00e9e\n".encode("utf-8"))

names = source.read_text(encoding="latin-1").splitlines()

print(names)
print("looks like valid text:", all(n.isprintable() for n in names))


['BjÃ¶rk', 'RenÃ©e']
looks like valid text: True


Two names, both printable, both wrong. Every check you might apply says the data is fine.

There is no way for a program to detect this from the values alone. The only defenses are
knowing the encoding, or looking at the output with your own eyes early enough to notice.

That second one is why printing the first few rows of any file you did not create is worth the
ten seconds it takes.


### Cleaning up


In [26]:
shutil.rmtree(scratch)

print("scratch still there:", scratch.exists())


scratch still there: False


## Recap

- Files hold **bytes**. An encoding is the rule for turning characters into bytes and back.
- Nothing inside a plain text file records which encoding was used.
- Decoding with the wrong rule usually **does not fail**; it produces different characters.
- `cafÃ©` is UTF-8 bytes read as Latin-1, and it can be repaired by encoding back and decoding
  again.
- `UnicodeDecodeError` names the codec, the byte and the position, and the position is the
  useful part.
- `errors="ignore"` drops characters silently. Prefer `strict`, or `replace` when you must
  continue.
- `latin-1` accepts any bytes, so it never tells you the guess was wrong.
- Excel writes a byte order mark; read those files with `encoding="utf-8-sig"`.
- Always pass `encoding=`. The default varies between machines.


## What is next

The **CSV** notebook, which reads the format most data arrives in. Everything in this notebook
applies to it, and the `csv` module adds problems of its own: quoting, embedded commas, and rows
that are not the length you expected.


---

&#8592; **Previous:** [Reading and Writing Text](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/files-paths-and-formats/02-reading-and-writing-text.ipynb)  &nbsp;·&nbsp;  [Files, Paths and Formats Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/files-paths-and-formats.html)
